In [7]:

# CELL 1 — ENVIRONMENT VERIFICATION & MPI4PY INSTALLATION

!nvidia-smi --query-gpu=name,uuid,compute_cap,memory.total --format=csv
!mpirun --version | head -n 2
!pip install -q mpi4py numba numpy


name, uuid, compute_cap, memory.total [MiB]
Tesla T4, GPU-8dd74487-7c4d-8b8a-3955-5bf6707b050d, 7.5, 15360 MiB
mpirun (Open MPI) 4.1.6



In [8]:

# CELL 2 — STUDENT IDENTITY CONFIGURATION

STUDENT_ID = "230103360"

# GitHub username will be added before final submission.
GITHUB_USERNAME = None

print("Student ID:", STUDENT_ID)
print("GitHub Username:", GITHUB_USERNAME)
print("Environment configuration initialized.")


Student ID: 230103360
GitHub Username: None
Environment configuration initialized.


In [9]:

# CELL 3 — PYTHON LIBRARIES & CUDA VERIFICATION

import numpy as np
import numba
from numba import cuda
from mpi4py import MPI

print("NumPy version:", np.__version__)
print("Numba version:", numba.__version__)
print("MPI version:", MPI.Get_version())

print("CUDA available:", cuda.is_available())

if cuda.is_available():
    device = cuda.get_current_device()
    print("GPU name:", device.name)
    print("GPU compute capability:", device.compute_capability)

print("\nEnvironment verification completed.")


NumPy version: 2.4.6
Numba version: 0.64.0
MPI version: (3, 1)
CUDA available: True
GPU name: Tesla T4
GPU compute capability: ComputeCapability(major=7, minor=5)

Environment verification completed.


In [13]:

%%writefile mpi_topology_test.py

from mpi4py import MPI

comm = MPI.COMM_WORLD
size = comm.Get_size()
rank = comm.Get_rank()

if size != 4:
    raise RuntimeError(f"Expected 4 MPI processes, got {size}")

# Create 2x2 Cartesian topology
dims = [2, 2]
periods = [False, False]

cart_comm = comm.Create_cart(
    dims=dims,
    periods=periods,
    reorder=False
)

coords = cart_comm.Get_coords(rank)

# Find neighboring processes
up, down = cart_comm.Shift(0, 1)
left, right = cart_comm.Shift(1, 1)

result = {
    "Rank": rank,
    "Coordinates": coords,
    "Up": up,
    "Down": down,
    "Left": left,
    "Right": right
}

results = comm.gather(result, root=0)

if rank == 0:
    print("===== MPI CARTESIAN TOPOLOGY =====")
    for item in results:
        print(item)
    print("===== TOPOLOGY CREATED SUCCESSFULLY =====")


Overwriting mpi_topology_test.py


In [14]:
!mpirun --allow-run-as-root --oversubscribe -np 4 python mpi_topology_test.py

In [15]:
!which mpirun
!mpirun --version
!ls -l mpi_topology_test.py

/usr/bin/mpirun
mpirun (Open MPI) 4.1.6

Report bugs to http://www.open-mpi.org/community/help/
-rw-r--r-- 1 root root 797 Nov 19 07:06 mpi_topology_test.py


In [18]:
!mpirun --allow-run-as-root --oversubscribe -np 4 python -u mpi_topology_test.py

In [19]:
!timeout 15s mpirun --allow-run-as-root --oversubscribe -np 4 python -c "from mpi4py import MPI; print('Hello from rank', MPI.COMM_WORLD.Get_rank(), 'of', MPI.COMM_WORLD.Get_size(), flush=True)"

print("TEST FINISHED")

TEST FINISHED


In [20]:
import subprocess

cmd = [
    "mpirun",
    "--allow-run-as-root",
    "--oversubscribe",
    "-np", "4",
    "python",
    "-c",
    "from mpi4py import MPI; print('Hello from rank', MPI.COMM_WORLD.Get_rank(), 'of', MPI.COMM_WORLD.Get_size(), flush=True)"
]

try:
    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        timeout=20
    )

    print("RETURN CODE:", result.returncode)
    print("STDOUT:", repr(result.stdout))
    print("STDERR:", repr(result.stderr))

except Exception as e:
    print(type(e).__name__, str(e))

RETURN CODE: 1
STDOUT: ''
STDERR: ''


In [22]:

import subprocess

cmd = [
    "/usr/bin/mpirun",
    "--allow-run-as-root",
    "--oversubscribe",
    "--mca", "btl", "self,tcp",
    "--mca", "pml", "ob1",
    "-np", "4",
    "/bin/echo", "MPI_WORKS"
]

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True,
    timeout=30
)

print("RETURN CODE:", result.returncode)
print("OUTPUT:", repr(result.stdout))
print("ERROR:", repr(result.stderr))


RETURN CODE: 1
OUTPUT: ''
ERROR: ''


In [23]:
import subprocess

cmd = [
    "/usr/bin/mpirun",
    "--allow-run-as-root",
    "--oversubscribe",
    "--mca", "plm", "isolated",
    "--mca", "btl", "self,tcp",
    "--mca", "pml", "ob1",
    "-np", "4",
    "/bin/echo", "MPI_WORKS"
]

try:
    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        timeout=30
    )
    print("RETURN CODE:", result.returncode)
    print("OUTPUT:", repr(result.stdout))
    print("ERROR:", repr(result.stderr))
except subprocess.TimeoutExpired:
    print("MPI LAUNCH TIMED OUT")

RETURN CODE: 1
OUTPUT: ''
ERROR: ''


In [24]:
!apt-get update -qq
!apt-get remove -y openmpi-bin libopenmpi-dev
!apt-get install -y mpich libmpich-dev
!python -m pip uninstall -y mpi4py
!python -m pip install --no-cache-dir mpi4py

!which mpirun
!mpirun --version

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following packages were automatically installed and are no longer required:
  ibverbs-providers libamd-comgr2 libamdhip64-5 libevent-2.1-7t64 libevent-dev
  libevent-extra-2.1-7t64 libevent-openssl-2.1-7t64 libevent-pthreads-2.1-7t64
  libfabric1 libhsa-runtime64-1 libhsakmt1 libhwloc-dev libhwloc-plugins
  libhwloc15 libibverbs-dev libibverbs1 libjs-jquery-ui libllvm17t64
  libmumps-headers-dev libmunge2 libnl-3-200 libnl-3-dev libnl-route-3-200
  libnl-route-3-dev libnuma-dev libopenmpi3t64 libpmix-dev libpmix2t64
  libpsm-infinipath1 libpsm2-2 librdmacm1t64 libucx0 libxnvctrl0
  openmpi-common
Use 'apt autoremove' to remove them.
The following packages will be REMOVED:
  coinor-libipopt-dev libmump

In [1]:
!mpirun -np 4 python -c "from mpi4py import MPI; print('Rank:', MPI.COMM_WORLD.Get_rank(), 'of', MPI.COMM_WORLD.Get_size())"

Rank: 0 of 1
Rank: 0 of 1
Rank: 0 of 1
Rank: 0 of 1


In [2]:
from mpi4py import MPI

print("MPI library:")
print(MPI.Get_library_version())

print("\nMPI size:", MPI.COMM_WORLD.Get_size())
print("MPI rank:", MPI.COMM_WORLD.Get_rank())

MPI library:
MPICH Version:      4.2.0
MPICH Release date: Fri Feb  9 12:29:21 CST 2024
MPICH ABI:          16:0:4
MPICH Device:       ch4:ucx
MPICH configure:    --build=x86_64-linux-gnu --prefix=/usr --includedir=${prefix}/include --mandir=${prefix}/share/man --infodir=${prefix}/share/info --sysconfdir=/etc --localstatedir=/var --disable-option-checking --disable-silent-rules --libdir=${prefix}/lib/x86_64-linux-gnu --runstatedir=/run --disable-maintainer-mode --disable-dependency-tracking --with-pmix=/usr/lib/x86_64-linux-gnu/pmix2 --with-ucx=/usr --with-device=ch4:ucx --with-pm=hydra --with-hwloc-prefix=/usr --with-wrapper-dl-type=none --enable-shared --without-yaksa --prefix=/usr --enable-fortran=all --disable-rpath --disable-wrapper-rpath --sysconfdir=/etc/mpich --libdir=${prefix}/lib/x86_64-linux-gnu/mpich/lib --includedir=${prefix}/lib/x86_64-linux-gnu/mpich/include --docdir=/usr/share/doc/mpich CPPFLAGS= CFLAGS= CXXFLAGS= FFLAGS=-O2 -fno-omit-frame-pointer -mno-omit-leaf-frame-

In [3]:
import subprocess

cmd = [
    "mpirun", "-np", "4",
    "python", "-c",
    "import os; from mpi4py import MPI; "
    "print('RANK ENV:', os.environ.get('PMI_RANK'), "
    "os.environ.get('PMIX_RANK'), "
    "os.environ.get('PMI_SIZE'), "
    "'MPI:', MPI.COMM_WORLD.Get_rank(), "
    'MPI.COMM_WORLD.Get_size(), flush=True)'
]

result = subprocess.run(cmd, capture_output=True, text=True, timeout=30)

print("RETURN CODE:", result.returncode)
print("STDOUT:\n", result.stdout)
print("STDERR:\n", result.stderr)

RETURN CODE: 0
STDOUT:
 RANK ENV: 3 None 4 MPI: 0 1
RANK ENV: 0 None 4 MPI: 0 1
RANK ENV: 2 None 4 MPI: 0 1
RANK ENV: 1 None 4 MPI: 0 1

STDERR:
 


In [4]:
import os
import subprocess
from mpi4py import MPI

print("MPI extension:", MPI.__file__)
print("\nMPI environment variables:")
for key in sorted(os.environ):
    if any(x in key for x in ["PMI", "PMIX", "HYDRA", "MPI"]):
        print(f"{key} = {os.environ[key]}")

print("\nLoaded MPI libraries:")
result = subprocess.run(
    ["ldd", MPI.__file__],
    capture_output=True,
    text=True
)
print(result.stdout)

MPI extension: /usr/local/lib/python3.13/dist-packages/mpi4py/MPI.mpich.cpython-313-x86_64-linux-gnu.so

MPI environment variables:

Loaded MPI libraries:
	linux-vdso.so.1 (0x00007fff121f4000)
	libmpi.so.12 => /lib/x86_64-linux-gnu/libmpi.so.12 (0x00007e8b6bcb0000)
	libpthread.so.0 => /lib/x86_64-linux-gnu/libpthread.so.0 (0x00007e8b6bcab000)
	libc.so.6 => /lib/x86_64-linux-gnu/libc.so.6 (0x00007e8b6ba98000)
	libhwloc.so.15 => /usr/local/lib/libhwloc.so.15 (0x00007e8b6b800000)
	libpmix.so.2 => /lib/x86_64-linux-gnu/libpmix.so.2 (0x00007e8b6b5ee000)
	libucp.so.0 => /lib/x86_64-linux-gnu/libucp.so.0 (0x00007e8b6b48b000)
	libucs.so.0 => /lib/x86_64-linux-gnu/libucs.so.0 (0x00007e8b6b425000)
	/lib64/ld-linux-x86-64.so.2 (0x00007e8b6e5a2000)
	libm.so.6 => /lib/x86_64-linux-gnu/libm.so.6 (0x00007e8b6b33c000)
	libmunge.so.2 => /lib/x86_64-linux-gnu/libmunge.so.2 (0x00007e8b6ba8e000)
	libevent_core-2.1.so.7 => /lib/x86_64-linux-gnu/libevent_core-2.1.so.7 (0x00007e8b6b307000)
	libevent_pthreads

In [5]:
import subprocess

code = """
import os
from mpi4py import MPI

print("ENV:", {
    k: v for k, v in os.environ.items()
    if any(x in k for x in ["PMI", "PMIX", "HYDRA"])
}, flush=True)

print(
    "MPI:",
    MPI.COMM_WORLD.Get_rank(),
    "of",
    MPI.COMM_WORLD.Get_size(),
    flush=True
)
"""

result = subprocess.run(
    ["mpirun", "-np", "4", "python", "-c", code],
    capture_output=True,
    text=True,
    timeout=30
)

print("RETURN CODE:", result.returncode)
print("OUTPUT:\n", result.stdout)
print("ERROR:\n", result.stderr)

RETURN CODE: 0
OUTPUT:
 ENV: {'PMI_SIZE': '4', 'PMI_FD': '9', 'PMI_RANK': '0'}
MPI: 0 of 1
ENV: {'PMI_SIZE': '4', 'PMI_FD': '10', 'PMI_RANK': '1'}
MPI: 0 of 1
ENV: {'PMI_SIZE': '4', 'PMI_FD': '19', 'PMI_RANK': '3'}
MPI: 0 of 1
ENV: {'PMI_SIZE': '4', 'PMI_FD': '15', 'PMI_RANK': '2'}
MPI: 0 of 1

ERROR:
 


In [6]:
!mpichversion | head -n 15
!mpirun -info | head -n 30

MPICH Version:      4.2.0
MPICH Release date: Fri Feb  9 12:29:21 CST 2024
MPICH ABI:          16:0:4
MPICH Device:       ch4:ucx
MPICH configure:    --build=x86_64-linux-gnu --prefix=/usr --includedir=${prefix}/include --mandir=${prefix}/share/man --infodir=${prefix}/share/info --sysconfdir=/etc --localstatedir=/var --disable-option-checking --disable-silent-rules --libdir=${prefix}/lib/x86_64-linux-gnu --runstatedir=/run --disable-maintainer-mode --disable-dependency-tracking --with-pmix=/usr/lib/x86_64-linux-gnu/pmix2 --with-ucx=/usr --with-device=ch4:ucx --with-pm=hydra --with-hwloc-prefix=/usr --with-wrapper-dl-type=none --enable-shared --without-yaksa --prefix=/usr --enable-fortran=all --disable-rpath --disable-wrapper-rpath --sysconfdir=/etc/mpich --libdir=${prefix}/lib/x86_64-linux-gnu/mpich/lib --includedir=${prefix}/lib/x86_64-linux-gnu/mpich/include --docdir=/usr/share/doc/mpich CPPFLAGS= CFLAGS= CXXFLAGS= FFLAGS=-O2 -fno-omit-frame-pointer -mno-omit-leaf-frame-pointer -ffil

In [7]:
import subprocess

cmd = [
    "mpirun",
    "-np", "4",
    "python", "-c",
    "from mpi4py import MPI; "
    "print('Rank:', MPI.COMM_WORLD.Get_rank(), "
    "'of', MPI.COMM_WORLD.Get_size(), flush=True)"
]

env = __import__("os").environ.copy()
env["MPIR_CVAR_CH4_RUNTIME_CONF"] = "enable"

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True,
    timeout=30,
    env=env
)

print("RETURN CODE:", result.returncode)
print("OUTPUT:\n", result.stdout)
print("ERROR:\n", result.stderr)

RETURN CODE: 0
OUTPUT:
 Rank: 0 of 1
Rank: 0 of 1
Rank: 0 of 1
Rank: 0 of 1

ERROR:
 


In [8]:
!which mpicc
!mpicc -show

/usr/bin/mpicc
gcc -Wl,-Bsymbolic-functions -flto=auto -ffat-lto-objects -Wl,-z,relro -I/usr/lib/x86_64-linux-gnu/mpich/include -L/usr/lib/x86_64-linux-gnu/mpich/lib -lmpich


In [9]:
!python -m pip uninstall -y mpi4py
!python -m pip cache remove mpi4py

import os
os.environ["MPICC"] = "/usr/bin/mpicc"

!python -m pip install --no-cache-dir --no-binary=mpi4py mpi4py

Found existing installation: mpi4py 4.1.2
Uninstalling mpi4py-4.1.2:
  Successfully uninstalled mpi4py-4.1.2
Files removed: 0 (0 bytes)
Directories removed: 0
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for mpi4py: filename=mpi4py-4.1.2-cp313-cp313-linux_x86_64.whl size=4896569 sha256=24d5a9779c21839a766d7cafda0e23577dba4277e0e471da9f3ba42a3c3dbf4f
  Stored in directory: /tmp/pip-ephem-wheel-cache-3kda0m0j/wheels/64/18/2b/672c904baeadaf97a4e8c850f4c3b59f44216fd44263094e7c
Successfully built mpi4py


In [1]:
!mpirun -np 4 python -c "from mpi4py import MPI; print('Rank:', MPI.COMM_WORLD.Get_rank(), 'of', MPI.COMM_WORLD.Get_size(), flush=True)"

Rank: 0 of 1
Rank: 0 of 1
Rank: 0 of 1
Rank: 0 of 1


In [2]:
!which mpirun
!mpirun --version
!which mpicc
!python -m pip show mpi4py
!ldconfig -p | grep libmpi

/usr/bin/mpirun
HYDRA build details:
    Version:                                 4.2.0
    Release Date:                            Fri Feb  9 12:29:21 CST 2024
    CC:                              gcc -Wdate-time -D_FORTIFY_SOURCE=3 -g -O2 -fno-omit-frame-pointer -mno-omit-leaf-frame-pointer -ffile-prefix-map=/build/mpich-C3oteG/mpich-4.2.0=. -flto=auto -ffat-lto-objects -fstack-protector-strong -fstack-clash-protection -Wformat -Werror=format-security -fcf-protection -fdebug-prefix-map=/build/mpich-C3oteG/mpich-4.2.0=/usr/src/mpich-4.2.0-5build3 -Wl,-Bsymbolic-functions -flto=auto -ffat-lto-objects -Wl,-z,relro 
    Configure options:                       '--with-hwloc-prefix=/usr' '--with-device=ch4:ucx' 'FFLAGS=-O2 -fno-omit-frame-pointer -mno-omit-leaf-frame-pointer -ffile-prefix-map=/build/mpich-C3oteG/mpich-4.2.0=. -flto=auto -ffat-lto-objects -fstack-protector-strong -fstack-clash-protection -fcf-protection -fdebug-prefix-map=/build/mpich-C3oteG/mpich-4.2.0=/usr/src/mpich-4.2

In [3]:
import subprocess

code = """
import os
from mpi4py import MPI

fd = os.environ.get("PMI_FD")

try:
    fd_status = os.fstat(int(fd)) if fd is not None else "PMI_FD missing"
except Exception as e:
    fd_status = repr(e)

print(
    "PMI_RANK:", os.environ.get("PMI_RANK"),
    "| PMI_FD:", fd,
    "| FD_STATUS:", fd_status,
    "| MPI:", MPI.COMM_WORLD.Get_rank(),
    "of", MPI.COMM_WORLD.Get_size(),
    flush=True
)
"""

result = subprocess.run(
    ["mpirun", "-np", "4", "python", "-c", code],
    capture_output=True,
    text=True,
    timeout=30
)

print("OUTPUT:\n", result.stdout)
print("ERROR:\n", result.stderr)

OUTPUT:
 PMI_RANK: 2 | PMI_FD: 15 | FD_STATUS: os.stat_result(st_mode=49663, st_ino=232522, st_dev=8, st_nlink=1, st_uid=0, st_gid=0, st_size=0, st_atime=0, st_mtime=0, st_ctime=0) | MPI: 0 of 1
PMI_RANK: 0 | PMI_FD: 9 | FD_STATUS: os.stat_result(st_mode=49663, st_ino=232512, st_dev=8, st_nlink=1, st_uid=0, st_gid=0, st_size=0, st_atime=0, st_mtime=0, st_ctime=0) | MPI: 0 of 1
PMI_RANK: 1 | PMI_FD: 10 | FD_STATUS: os.stat_result(st_mode=49663, st_ino=232517, st_dev=8, st_nlink=1, st_uid=0, st_gid=0, st_size=0, st_atime=0, st_mtime=0, st_ctime=0) | MPI: 0 of 1
PMI_RANK: 3 | PMI_FD: 19 | FD_STATUS: os.stat_result(st_mode=49663, st_ino=232527, st_dev=8, st_nlink=1, st_uid=0, st_gid=0, st_size=0, st_atime=0, st_mtime=0, st_ctime=0) | MPI: 0 of 1

ERROR:
 


In [4]:
%%writefile mpi_check.c
#include <stdio.h>
#include <mpi.h>

int main(int argc, char** argv) {
    MPI_Init(&argc, &argv);

    int rank, size;
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);

    printf("C MPI Rank: %d of %d\n", rank, size);

    MPI_Finalize();
    return 0;
}

Writing mpi_check.c


In [5]:
!mpicc mpi_check.c -o mpi_check
!mpirun -np 4 ./mpi_check

C MPI Rank: 0 of 1
C MPI Rank: 0 of 1
C MPI Rank: 0 of 1
C MPI Rank: 0 of 1


In [7]:
!apt-get purge -y mpich libmpich-dev
!apt-get autoremove -y
!apt-get update -qq
!apt-get install -y openmpi-bin libopenmpi-dev

^C
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
0 upgraded, 0 newly installed, 0 to remove and 100 not upgraded.
^C
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  ibverbs-providers javascript-common libamd-comgr2 libamdhip64-5
  libcaf-openmpi-3t64 libcoarrays-dev libcoarrays-openmpi-dev
  libevent-2.1-7t64 libevent-dev libevent-extra-2.1-7t64
  libevent-openssl-2.1-7t64 libevent-pthreads-2.1-7t64 libfabric1
  libhsa-runtime64-1 libhsakmt1 libhwloc-dev libhwloc-plugins libhwloc15
  libibverbs-dev libibverbs1 libjs-jquery-ui libllvm17t64 libmunge2
  libnl-3-200 libnl-3-dev libnl-route-3-200 libnl-route-3-dev libnuma-dev
  libopenmpi3t64 libpmix-dev libpmix2t64 libpsm-infinipath1 libpsm2-2
  librdmacm1t64 libucx0 libxnvctrl0 openmpi-common
Suggested packages:
  apache2 | lighttpd | httpd libhwloc-contrib-plugins libjs-jquery-ui-docs


In [8]:
!apt-get install -y openmpi-bin libopenmpi-dev

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  ibverbs-providers javascript-common libamd-comgr2 libamdhip64-5
  libcaf-openmpi-3t64 libcoarrays-dev libcoarrays-openmpi-dev
  libevent-2.1-7t64 libevent-dev libevent-extra-2.1-7t64
  libevent-openssl-2.1-7t64 libevent-pthreads-2.1-7t64 libfabric1
  libhsa-runtime64-1 libhsakmt1 libhwloc-dev libhwloc-plugins libhwloc15
  libibverbs-dev libibverbs1 libjs-jquery-ui libllvm17t64 libmunge2
  libnl-3-200 libnl-3-dev libnl-route-3-200 libnl-route-3-dev libnuma-dev
  libopenmpi3t64 libpmix-dev libpmix2t64 libpsm-infinipath1 libpsm2-2
  librdmacm1t64 libucx0 libxnvctrl0 openmpi-common
Suggested packages:
  apache2 | lighttpd | httpd libhwloc-contrib-plugins libjs-jquery-ui-docs
  openmpi-doc
The following NEW packages will be installed:
  ibverbs-providers javascript-common libamd-comgr2 libamdhip64-5
  libcaf-openmpi-3t64 libcoarrays-dev libc

In [9]:
   import numpy as np
from numba import cuda

# Local domain: 256x256 + 1 ghost layer on each side
N = 256
SIZE = N + 2

@cuda.jit
def pack_halos(u, left, right, top, bottom):
    i = cuda.grid(1)
    if i < N:
        left[i]   = u[i + 1, 1]
        right[i]  = u[i + 1, N]
        top[i]    = u[1, i + 1]
        bottom[i] = u[N, i + 1]

@cuda.jit
def unpack_halos(u, left, right, top, bottom):
    i = cuda.grid(1)
    if i < N:
        u[i + 1, 0]   = left[i]
        u[i + 1, N+1] = right[i]
        u[0, i + 1]   = top[i]
        u[N+1, i + 1] = bottom[i]

# Test allocation on GPU
u = cuda.to_device(np.zeros((SIZE, SIZE), dtype=np.float64))

left = cuda.device_array(N, dtype=np.float64)
right = cuda.device_array(N, dtype=np.float64)
top = cuda.device_array(N, dtype=np.float64)
bottom = cuda.device_array(N, dtype=np.float64)

threads = 256
blocks = (N + threads - 1) // threads

pack_halos[blocks, threads](u, left, right, top, bottom)
cuda.synchronize()

print("CUDA halo buffers created successfully.")
print("Local array:", u.shape)
print("GPU:", cuda.get_current_device().name)

/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


CUDA halo buffers created successfully.
Local array: (258, 258)
GPU: Tesla T4


In [12]:
import math

@cuda.jit
def stencil_kernel(u, kappa, source, out, dt):
    tile = cuda.shared.array((18, 18), dtype=numba.float64)
    ktile = cuda.shared.array((18, 18), dtype=numba.float64)

    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y
    tid = ty * 16 + tx

    # Load 18x18 tile cooperatively
    for idx in range(tid, 324, 256):
        sy = idx // 18
        sx = idx % 18

        gy = 1 + cuda.blockIdx.y * 16 + sy - 1
        gx = 1 + cuda.blockIdx.x * 16 + sx - 1

        tile[sy, sx] = u[gy, gx]
        ktile[sy, sx] = kappa[gy, gx]

    cuda.syncthreads()

    r = 1 + cuda.blockIdx.y * 16 + ty
    c = 1 + cuda.blockIdx.x * 16 + tx

    center = tile[ty+1, tx+1]
    kc = ktile[ty+1, tx+1]

    ke = 0.5 * (kc + ktile[ty+1, tx+2])
    kw = 0.5 * (kc + ktile[ty+1, tx])
    kn = 0.5 * (kc + ktile[ty, tx+1])
    ks = 0.5 * (kc + ktile[ty+2, tx+1])

    lap = (
        ke * (tile[ty+1, tx+2] - center)
        - kw * (center - tile[ty+1, tx])
        + ks * (tile[ty+2, tx+1] - center)
        - kn * (center - tile[ty, tx+1])
    )

    out[r, c] = center + dt * (lap - 0.15 * center + source[r, c])


# Initialize test arrays
u = cuda.to_device(np.random.rand(258, 258))
kappa = cuda.to_device(np.full((258, 258), 0.16))
source = cuda.to_device(np.zeros((258, 258)))
out = cuda.device_array((258, 258), dtype=np.float64)

threads = (16, 16)
blocks = (16, 16)

stencil_kernel[blocks, threads](u, kappa, source, out, 1e-6)
cuda.synchronize()

print("Stencil kernel executed successfully.")
print("Output shape:", out.shape)
print("GPU:", cuda.get_current_device().name)

Stencil kernel executed successfully.
Output shape: (258, 258)
GPU: Tesla T4


In [11]:
import numba
print("Numba imported:", numba.__version__)

Numba imported: 0.64.0


In [13]:
# Initialize local subdomain
host_u = np.zeros((258, 258), dtype=np.float64)
host_u[1:257, 1:257] = np.random.default_rng(42).random((256, 256))

u = cuda.to_device(host_u)
out = cuda.device_array((258, 258), dtype=np.float64)

kappa = cuda.to_device(np.full((258, 258), 0.16))
source = cuda.to_device(np.zeros((258, 258)))

dt = 1e-6
energy_trace = []

# Run 400 time steps
for step in range(1, 401):
    stencil_kernel[(16, 16), (16, 16)](
        u, kappa, source, out, dt
    )
    u, out = out, u

    if step % 100 == 0:
        cuda.synchronize()
        result = u.copy_to_host()
        energy = np.sum(result[1:257, 1:257] ** 2)
        energy_trace.append((step, energy))
        print(f"Step {step}: energy = {energy:.8f}")

print("\n400 CUDA iterations completed.")

Step 100: energy = 21910.63471545
Step 200: energy = 21909.26886911
Step 300: energy = 21907.90319768
Step 400: energy = 21906.53770113

400 CUDA iterations completed.


In [14]:
@cuda.jit
def zero_boundaries(u):
    i = cuda.grid(1)
    if i < 258:
        u[0, i] = 0.0
        u[257, i] = 0.0
        u[i, 0] = 0.0
        u[i, 257] = 0.0

zero_boundaries[2, 256](u)
cuda.synchronize()

print("Physical boundary conditions applied.")

Physical boundary conditions applied.


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 2 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [15]:
# Reinitialize both buffers
host_u = np.zeros((258, 258), dtype=np.float64)
host_u[1:257, 1:257] = np.random.default_rng(42).random((256, 256))

u = cuda.to_device(host_u)
out = cuda.to_device(np.zeros((258, 258), dtype=np.float64))

kappa = cuda.to_device(np.full((258, 258), 0.16))
source = cuda.to_device(np.zeros((258, 258), dtype=np.float64))

energy_trace = []
dt = 1e-6

for step in range(1, 401):
    stencil_kernel[(16, 16), (16, 16)](
        u, kappa, source, out, dt
    )

    zero_boundaries[(2,), (256,)](out)
    u, out = out, u

    if step % 100 == 0:
        cuda.synchronize()
        result = u.copy_to_host()
        energy = np.sum(result[1:257, 1:257] ** 2)
        energy_trace.append((step, energy))
        print(f"Step {step}: energy = {energy:.8f}")

print("Corrected 400-step simulation completed.")

Step 100: energy = 21910.63471545
Step 200: energy = 21909.26886911
Step 300: energy = 21907.90319768
Step 400: energy = 21906.53770113
Corrected 400-step simulation completed.


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 2 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [16]:
# PTX audit
print("Compiled signatures:", stencil_kernel.signatures)

ptx = stencil_kernel.inspect_asm(stencil_kernel.signatures[0])

print("\nPTX instruction audit:")
print("ld.shared:", ptx.count("ld.shared"))
print("st.shared:", ptx.count("st.shared"))
print("bar.sync:", ptx.count("bar.sync"))

print("\nPTX audit completed.")

Compiled signatures: [(Array(float64, 2, 'C', False, aligned=True), Array(float64, 2, 'C', False, aligned=True), Array(float64, 2, 'C', False, aligned=True), Array(float64, 2, 'C', False, aligned=True), float64)]

PTX instruction audit:
ld.shared: 10
st.shared: 10
bar.sync: 1

PTX audit completed.


In [18]:
import json

STUDENT_ID = "230103360"

manifest = {
    "student_id": STUDENT_ID,
    "gpu": "Tesla T4",
    "compute_capability": "7.5",
    "local_domain": [256, 256],
    "padded_domain": [258, 258],
    "cuda_iterations": 400,
    "time_step": 1e-6,
    "energy_trace": [
        {"step": step, "energy": float(energy)}
        for step, energy in energy_trace
    ],
    "ptx_audit": {
        "ld.shared": 10,
        "st.shared": 10,
        "bar.sync": 1
    },
    "mpi_processes_verified": False,
    "status": "CUDA validation completed; MPI validation pending"
}

with open("submission_manifest.json", "w") as f:
    json.dump(manifest, f, indent=4)

print("submission_manifest.json successfully created!")

submission_manifest.json successfully created!
